# Global Superstore EDA

In this notebook, we explore the cleaned Global Superstore data stored in MySQL. The goal is not only to calculate business metrics, but also to explain the reasoning behind each step so that the workflow is easy to follow from raw extraction to business insight.

The business questions covered in this notebook focus on customer behavior, product performance, geographic performance, delivery efficiency, and overall profitability.


## 1. Imports and database connection

In [1]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 2. Load dataset

Data preparation has already been done, so we load the analysis prepared datasets.  



In [ ]:
query = '''
SELECT
    row_id,
    order_id,
    order_date,
    ship_date,
    ship_mode,
    customer_id,
    customer_name,
    segment,
    city,
    state,
    country,
    market,
    region,
    product_id,
    category,
    sub_category,
    product_name,
    sales,
    quantity,
    discount,
    profit,
    shipping_cost,
    order_priority
FROM g_superstore_sorted
'''

df = pd.read_sql(query, engine)
df.head()


## 3. Data structure and type checks

Confirm the number of rows, the available fields, and whether the imported datatypes match the expected business meaning of each column.


In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])
df["sales"] = pd.to_numeric(df["sales"])
df["quantity"] = pd.to_numeric(df["quantity"])
df["discount"] = pd.to_numeric(df["discount"])
df["profit"] = pd.to_numeric(df["profit"])
df["shipping_cost"] = pd.to_numeric(df["shipping_cost"])

print(df.info())
print(df.isnull().sum())
df.describe(include="all")


## 4. Feature engineering

To answer the business questions more effectively, we derive a few additional columns.

- `year` and `month` make time-based analysis easier.
- `delivery_days` measures the time between order placement and shipping.
- `unit_price` provides an approximate price-per-unit view.
- `profit_margin_pct` standardizes profitability relative to sales rather than absolute profit alone.

These variables make the later customer, pricing, and delivery analyses more interpretable.


In [ ]:
df["year"] = df["order_date"].dt.year
df["month"] = df["order_date"].dt.month
df["delivery_days"] = (df["ship_date"] - df["order_date"]).dt.days
df["unit_price"] = np.where(df["quantity"] > 0, df["sales"] / df["quantity"], np.nan)
df["profit_margin_pct"] = np.where(df["sales"] != 0, (df["profit"] / df["sales"]) * 100, np.nan)

df.head()


## 5. KPI overview

We start the business analysis with a compact KPI table.  
This gives a high-level view of company performance before drilling into customer and product patterns.

The KPI block includes:
- total sales
- total profit
- overall profit margin
- number of unique orders
- number of unique customers
- average order value
- average delivery time


In [5]:
total_sales = df["sales"].sum()
total_profit = df["profit"].sum()
profit_margin = (total_profit / total_sales) * 100 if total_sales != 0 else np.nan
total_orders = df["order_id"].nunique()
total_customers = df["customer_id"].nunique()
avg_order_value = total_sales / total_orders if total_orders != 0 else np.nan
avg_delivery_days = df["delivery_days"].mean()

kpi_df = pd.DataFrame({
    "KPI": [
        "Total Sales",
        "Total Profit",
        "Profit Margin (%)",
        "Unique Orders",
        "Unique Customers",
        "Average Order Value",
        "Average Delivery Time (days)"
    ],
    "Value": [
        total_sales,
        total_profit,
        profit_margin,
        total_orders,
        total_customers,
        avg_order_value,
        avg_delivery_days
    ]
})

kpi_df


,KPI,Value
0,Total Sales,"12,630,015.71"
1,Total Profit,"1,464,689.57"
2,Profit Margin (%),11.60
3,Unique Orders,"25,035.00"
4,Unique Customers,"1,590.00"
5,Average Order Value,504.49
6,Average Delivery Time (days),3.97


## 6. Customer analysis: purchase frequency

A major goal of this analysis is to understand whether frequent customers are driving the business.

To do that, we first aggregate transactions at the customer level. The frequency is measured **per customer**,

Once the customer-level summary is ready, we assign customers into purchase-frequency buckets.


In [6]:
customer_summary = (
    df.groupby(["customer_id", "customer_name", "segment"], as_index=False)
      .agg(
          total_orders=("order_id", "nunique"),
          total_sales=("sales", "sum"),
          total_profit=("profit", "sum")
      )
)

customer_summary["frequency_bucket"] = pd.qcut(
    customer_summary["total_orders"].rank(method="first"),
    q=4,
    labels=["Low", "Medium", "High", "Very High"]
)

customer_summary.head()


,customer_id,customer_name,segment,total_orders,total_sales,total_profit,frequency_bucket
0,AA-10315,Alex Avila,Consumer,19,"13,747.42",447.71,High
1,AA-10375,Allen Armold,Consumer,23,"5,884.19",677.48,High
2,AA-10480,Andrew Allen,Consumer,19,"17,563.45","1,504.63",High
3,AA-10645,Anna Andreadi,Consumer,36,"15,343.90","3,051.43",Very High
4,AA-315,Alex Avila,Consumer,7,"2,243.25",535.56,Medium


### Revenue, profit, and profit margin across customer frequency buckets

After defining the customer buckets, we summarize how much revenue and profit each group contributes.  
This helps answer whether high-frequency customers are not only buying more, but also remaining profitable.


In [7]:
freq_bucket_summary = (
    customer_summary.groupby("frequency_bucket", as_index=False)
    .agg(
        customers=("customer_id", "count"),
        total_orders=("total_orders", "sum"),
        total_sales=("total_sales", "sum"),
        total_profit=("total_profit", "sum")
    )
)

freq_bucket_summary["profit_margin_pct"] = (
    freq_bucket_summary["total_profit"] / freq_bucket_summary["total_sales"] * 100
)

freq_bucket_summary


/var/folders/pr/xym09mq17cs7tjmytw5dm2gm0000gn/T/ipykernel_19268/2267266878.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  customer_summary.groupby("frequency_bucket", as_index=False)


,frequency_bucket,customers,total_orders,total_sales,total_profit,profit_margin_pct
0,Low,398,1726,"595,403.14","51,985.65",8.73
1,Medium,397,3304,"1,063,567.02","98,852.60",9.29
2,High,397,8765,"4,668,206.05","556,163.52",11.91
3,Very High,398,11956,"6,302,839.50","757,687.80",12.02


In [8]:
fig = px.bar(
    freq_bucket_summary,
    x="frequency_bucket",
    y="total_sales",
    title="Revenue by Customer Frequency Bucket",
    text_auto=".2s",
    width=900,
    height=500
)
fig.show()


In [9]:
fig = px.bar(
    freq_bucket_summary,
    x="frequency_bucket",
    y="total_profit",
    title="Profit by Customer Frequency Bucket",
    text_auto=".2s",
    width=900,
    height=500
)
fig.show()


In [10]:
fig = px.bar(
    freq_bucket_summary,
    x="frequency_bucket",
    y="profit_margin_pct",
    title="Profit Margin by Customer Frequency Bucket",
    text_auto=".2f",
    width=900,
    height=500
)
fig.show()


## 7. Customer segment profitability by year

The next question is whether the profitability of customer segments changes over time.  
For this step, we aggregate sales and profit by year and segment, and then identify the top profit-generating segment for each year.

This allows us to see not only which segment performs best overall, but also whether the same segment stays dominant across multiple years.


In [11]:
segment_year_profit = (
    df.groupby(["year", "segment"], as_index=False)
      .agg(total_sales=("sales", "sum"), total_profit=("profit", "sum"))
)

segment_year_profit["profit_margin_pct"] = (
    segment_year_profit["total_profit"] / segment_year_profit["total_sales"] * 100
)

top_segment_each_year = (
    segment_year_profit.sort_values(["year", "total_profit"], ascending=[True, False])
    .groupby("year", as_index=False)
    .head(1)
)

top_segment_each_year


,year,segment,total_sales,total_profit,profit_margin_pct
0,2011,Consumer,"1,169,856.99","116,044.64",9.92
3,2012,Consumer,"1,459,966.55","165,351.07",11.33
6,2013,Consumer,"1,727,966.03","207,995.60",12.04
9,2014,Consumer,"2,140,328.86","257,606.19",12.04


In [12]:
fig = px.bar(
    segment_year_profit,
    x="year",
    y="total_profit",
    color="segment",
    barmode="group",
    title="Customer Segment Profitability by Year",
    width=700,
    height=300
)
fig.show()


## 8. Customer distribution across countries

Understanding the geographic spread of customers is useful for market prioritization.  
Here we count the number of unique customers per country and rank the countries from largest to smallest customer base.


In [13]:
customer_country_distribution = (
    df.groupby("country", as_index=False)
      .agg(unique_customers=("customer_id", "nunique"))
      .sort_values("unique_customers", ascending=False)
)

customer_country_distribution.head(15)


,country,unique_customers
139,United States,793
44,France,679
81,Mexico,670
6,Australia,660
47,Germany,581
26,China,549
138,United Kingdom,529
57,India,494
17,Brazil,472
58,Indonesia,469


In [14]:
fig = px.bar(
    customer_country_distribution.head(15),
    x="country",
    y="unique_customers",
    title="Top 15 Countries by Number of Customers",
    text_auto=True,
    width=900,
    height=500
)
fig.update_xaxes(tickangle=45)
fig.show()


## 9. Country-level sales performance

After looking at customer counts, we check which countries contribute the highest revenue.  
This helps distinguish between countries that simply have many customers and countries that actually generate strong business value.


In [15]:
country_sales = (
    df.groupby("country", as_index=False)
      .agg(total_sales=("sales", "sum"), total_profit=("profit", "sum"))
      .sort_values("total_sales", ascending=False)
)

country_sales.head(10)


,country,total_sales,total_profit
139,United States,"2,295,509.78","286,014.59"
6,Australia,"924,715.02","103,773.01"
44,France,"858,648.18","108,967.04"
26,China,"700,394.06","150,662.87"
47,Germany,"628,685.33","107,305.07"
81,Mexico,"619,230.66","102,468.40"
57,India,"589,650.19","129,071.75"
138,United Kingdom,"528,244.85","111,774.73"
58,Indonesia,"404,887.67","15,608.74"
17,Brazil,"361,106.41","30,090.49"


In [16]:
fig = px.bar(
    country_sales.head(10),
    x="country",
    y="total_sales",
    title="Top 10 Countries by Sales",
    text_auto=".2s",
    width=900,
    height=500
)
fig.update_xaxes(tickangle=45)
fig.show()


## 10. Product analysis: top profit-making product types by year

For this analysis, we aggregate profit by year and sub-category and then keep the top five profit-making product types in each year.

This reveals which product groups consistently create value and whether product profitability shifts over time.


In [17]:
subcategory_year_profit = (
    df.groupby(["year", "sub_category"], as_index=False)
      .agg(total_sales=("sales", "sum"), total_profit=("profit", "sum"))
)

top5_subcategories_each_year = (
    subcategory_year_profit.sort_values(["year", "total_profit"], ascending=[True, False])
    .groupby("year", as_index=False)
    .head(5)
)

top5_subcategories_each_year


,year,sub_category,total_sales,total_profit
13,2011,Phones,"333,563.30","52,630.65"
6,2011,Copiers,"216,367.95","30,375.14"
5,2011,Chairs,"285,449.61","29,955.17"
4,2011,Bookcases,"259,396.29","27,518.79"
1,2011,Appliances,"173,383.49","22,838.43"
23,2012,Copiers,"327,168.66","51,843.24"
30,2012,Phones,"364,016.37","45,223.11"
17,2012,Accessories,"172,397.73","33,507.12"
22,2012,Chairs,"294,099.16","28,653.64"
21,2012,Bookcases,"317,953.51","28,137.25"


In [18]:
fig = px.bar(
    top5_subcategories_each_year,
    x="year",
    y="total_profit",
    color="sub_category",
    barmode="group",
    title="Top 5 Profit-Making Product Types by Year",
    width=900,
    height=500
)
fig.show()


## 11. Price versus sales at the day level

A possible business questions is whether lower prices are associated with higher sales.

To investigate that, we calculate the average unit price per day together with total daily sales.  
This gives a day-level view of the relationship between price and revenue.  


In [19]:
daily_price_sales = (
    df.groupby("order_date", as_index=False)
      .agg(
          avg_unit_price=("unit_price", "mean"),
          total_sales=("sales", "sum")
      )
      .sort_values("order_date")
)

daily_price_sales.head()


,order_date,avg_unit_price,total_sales
0,2011-01-01,54.35,808.57
1,2011-01-02,314.22,314.22
2,2011-01-03,91.07,"4,503.54"
3,2011-01-04,40.77,"2,808.87"
4,2011-01-05,86.52,"3,662.31"


In [20]:
fig = px.scatter(
    daily_price_sales,
    x="avg_unit_price",
    y="total_sales",
    title="Daily Average Price vs Daily Sales",
    width=900,
    height=500
)
fig.show()


## 12. Delivery performance across countries 
#### (Delivery efficiency as an operational metric)
We calculate the average number of delivery days for each country and then rank the countries to see where delivery is slowest.


In [21]:
country_delivery = (
    df.groupby("country", as_index=False)
      .agg(avg_delivery_days=("delivery_days", "mean"))
      .sort_values("avg_delivery_days", ascending=False)
)

country_delivery.head(20)


,country,avg_delivery_days
118,South Sudan,5.50
5,Armenia,5.33
75,Macedonia,5.25
64,Jamaica,5.03
39,Equatorial Guinea,5.00
127,Tajikistan,5.00
19,Burundi,5.00
80,Mauritania,4.90
84,Montenegro,4.75
55,Hong Kong,4.71


In [22]:
fig = px.bar(
    country_delivery.head(20),
    x="country",
    y="avg_delivery_days",
    title="Average Delivery Time Across Countries",
    text_auto=".2f",
    width=900,
    height=500
)
fig.update_xaxes(tickangle=45)
fig.show()


## 13. Profit analysis by category

Absolute profit alone is not enough when comparing categories, so we also calculate profit margin.  
This allows us to compare categories on a standardized basis and identify whether some categories are generating profit more efficiently than others.


In [23]:
category_profit = (
    df.groupby("category", as_index=False)
      .agg(
          total_sales=("sales", "sum"),
          total_profit=("profit", "sum"),
          avg_discount=("discount", "mean")
      )
)

category_profit["profit_margin_pct"] = (
    category_profit["total_profit"] / category_profit["total_sales"] * 100
)

category_profit.sort_values("total_profit", ascending=False)


,category,total_sales,total_profit,avg_discount,profit_margin_pct
2,Technology,"4,739,881.78","662,184.94",0.14,13.97
1,Office Supplies,"3,781,098.29","517,388.60",0.14,13.68
0,Furniture,"4,109,035.64","285,116.03",0.17,6.94


In [24]:
fig = px.bar(
    category_profit,
    x="category",
    y="total_profit",
    title="Profit by Category",
    text_auto=".2s",
    width=900,
    height=500
)
fig.show()


In [25]:
fig = px.bar(
    category_profit,
    x="category",
    y="profit_margin_pct",
    title="Profit Margin by Category",
    text_auto=".2f",
    width=900,
    height=500
)
fig.show()


## 14. Regional profitability

At this stage, we move from country-level performance to broader regional performance.  
This helps reveal whether some regions are systematically more profitable than others.


In [26]:
region_profit = (
    df.groupby("region", as_index=False)
      .agg(total_sales=("sales", "sum"), total_profit=("profit", "sum"))
)

region_profit["profit_margin_pct"] = (
    region_profit["total_profit"] / region_profit["total_sales"] * 100
)

region_profit.sort_values("total_profit", ascending=False)


,region,total_sales,total_profit,profit_margin_pct
3,Central,"2,821,828.27","311,313.54",11.03
7,North,"1,244,214.14","194,198.89",15.61
8,North Asia,"848,141.98","165,558.01",19.52
10,South,"1,599,493.88","140,115.80",8.76
4,Central Asia,"749,620.57","131,324.14",17.52
9,Oceania,"1,099,663.80","119,954.68",10.91
12,West,"725,355.59","108,404.72",14.95
6,East,"677,906.49","91,353.94",13.48
0,Africa,"783,049.86","88,588.38",11.31
5,EMEA,"806,056.41","43,865.80",5.44


In [27]:
fig = px.bar(
    region_profit,
    x="region",
    y="total_profit",
    title="Profit by Region",
    text_auto=".2s",
    width=900,
    height=500
)
fig.show()


## 15. Top loss-making products

To complement the profitability analysis, we identify the products with the lowest total profit.  
This is useful because a business can grow revenue while still being dragged down by a relatively small set of consistently loss-making products.


In [28]:
loss_products = (
    df.groupby(["product_id", "product_name"], as_index=False)
      .agg(total_sales=("sales", "sum"), total_profit=("profit", "sum"))
      .sort_values("total_profit", ascending=True)
      .head(10)
)

loss_products


,product_id,product_name,total_sales,total_profit
9574,TEC-MA-10000418,Cubify CubeX 3D Printer Double Head Print,"11,099.96","-8,879.97"
2614,OFF-AP-10001623,"Hoover Stove, White","11,728.84","-4,958.16"
9605,TEC-MA-10000822,Lexmark MX611dhe Monochrome Laser Printer,"16,829.90","-4,589.97"
10424,TEC-PH-10002991,"Apple Smart Phone, Full Size","7,259.16","-4,574.65"
9984,TEC-MOT-10003050,"Motorola Smart Phone, Cordless","10,348.75","-3,998.69"
9876,TEC-MA-10004125,Cubify CubeX 3D Printer Triple Head Print,"7,999.98","-3,839.99"
699,FUR-CH-10001582,"Office Star Executive Leather Armchair, Black","6,497.29","-3,066.78"
1993,FUR-TA-10000198,Chromcraft Bull-Nose Wood Oval Conference Tabl...,"9,917.64","-2,876.11"
2157,FUR-TA-10002885,"Lesro Computer Table, Fully Assembled","1,199.35","-2,798.49"
2118,FUR-TA-10002172,"Hon Conference Table, Rectangular","3,667.01","-2,619.31"


In [29]:
fig = px.bar(
    loss_products,
    x="product_name",
    y="total_profit",
    title="Top 10 Loss-Making Products",
    text_auto=".2s",
    width=900,
    height=500
)
fig.update_xaxes(tickangle=45)
fig.show()


## Summary of findings

At a high level, the analysis shows:
- how overall revenue and profit are distributed across the business,
- whether frequent customers are the strongest value drivers,
- which customer segments are most profitable over time,
- which countries and regions generate the most value,
- which product types lead yearly profitability,
- how delivery performance varies by country,
- and which products create the largest losses.


